In [ ]:
# 1. imports
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving sample.csv to sample (1).csv


In [ ]:
sample = pd.read_csv("sample.csv")

In [ ]:
sample['review/score'].value_counts()
sample['review/score'].value_counts(normalize=True)

,proportion
review/score,
5.0,0.60280
4.0,0.19678
3.0,0.08202
1.0,0.06720
2.0,0.05120


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.array([0, 1, 2])

sample['label'] = (
    sample['review/score'].astype(float).astype(int)
    .map({1: 0, 2: 0, 3: 1, 4: 2, 5: 2})
)

print(sample['label'].value_counts().sort_index())

weights = compute_class_weight('balanced', classes=classes, y=sample['label'])
print(weights)

label
0     5920
1     4101
2    39979
Name: count, dtype: int64
[2.81531532 4.06404942 0.41688553]


In [ ]:
sample['review/text'].str.len().describe()

,review/text
count,50000.000000
mean,826.884440
std,957.752815
min,6.000000
25%,263.000000
50%,522.000000
75%,1015.000000
max,22009.000000


In [ ]:
sample.isna().sum()

,0
review/score,0
review/text,0
label,0


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    sample, test_size=0.2, stratify=sample['label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

In [ ]:
from datasets import Dataset

train_ds = Dataset.from_pandas(train_df)
val_ds = Dataset.from_pandas(val_df)
test_ds = Dataset.from_pandas(test_df)

In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def tokenize_fn(batch):
    return tokenizer(batch["review/text"], truncation=True, max_length=128)

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
print(len(train_df), len(val_df), len(test_df))

40000 5000 5000


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch

class_weights = compute_class_weight(
    'balanced',
    classes=np.array([0, 1, 2]),
    y=train_df['label']
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

In [ ]:
import torch.nn as nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=3)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments
args = TrainingArguments(
    output_dir="/content/drive/MyDrive/nlp_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average='macro'),
    }

In [ ]:
# 4. collator + trainer
collator = DataCollatorWithPadding(tokenizer)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.678131,0.709863,0.806600,0.632242
2,0.516926,0.803239,0.821800,0.645804
3,0.453523,0.984517,0.828000,0.649032


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7500, training_loss=0.5862446783701579, metrics={'train_runtime': 522.6196, 'train_samples_per_second': 229.612, 'train_steps_per_second': 14.351, 'total_flos': 3974092830720000.0, 'train_loss': 0.5862446783701579, 'epoch': 3.0})

In [ ]:
results = trainer.evaluate(eval_dataset=test_ds)
print(results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.453523,0.921614,3,0.837800,0.669166


{'eval_loss': 0.9216141700744629, 'eval_accuracy': 0.8378, 'eval_f1': 0.6691655223871624}


In [ ]:
predictions = trainer.predict(test_ds)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(labels, preds)
print(cm)

[[ 427  110   55]
 [  70  196  144]
 [ 115  317 3566]]


In [ ]:
trainer.save_model('./final_model')
tokenizer.save_pretrained('./final_model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final_model/tokenizer_config.json', './final_model/tokenizer.json')

In [ ]:
!zip -r final_model.zip final_model

  adding: final_model/ (stored 0%)
  adding: final_model/model.safetensors (deflated 8%)
  adding: final_model/tokenizer_config.json (deflated 43%)
  adding: final_model/training_args.bin (deflated 53%)
  adding: final_model/tokenizer.json (deflated 71%)
  adding: final_model/config.json (deflated 51%)


In [ ]:
from google.colab import files
files.download('final_model.zip')

Found zip file at: /content/final_model.zip. Initializing download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>